# Витрина транзакций ТЭ — 3. Терминал

AI-14134. Одна строка = один терминал из факта. Таблица `sandbox_ai.tmp_shestopalov_acq_terminals`. Сначала должна быть собрана `01_te_trx.ipynb`.

Последняя запись терминала и модель устройства взяты из DAG витрины (`acq_04a_amortization.sql`).

- `version` — модель устройства из `dwh.bmrt_bm_det_device`.
- `date_on` — дата установки последней записи терминала.
- `date_off` — дата снятия последней записи. Пусто, если терминал не снят.


In [ ]:
from IPython.display import display
from rail_connectors.connection import connect

imp = connect(
    to="IMPALA",
    extra_options={"db": "sandbox_ai"},
    driver_args={"tez.queue.name": "ai"},
    kerberos={
        "keytab_path": "/home/jovyan/test_requests/tech.keytab",
        "use_credentials": True,
        "update_keytab": True,
    },
    user_params={"user_name": "Shestopalov-VYur"},
)
imp._init_connection()


In [ ]:
with imp:
    fact = imp.fetch("select count(*) as cnt from sandbox_ai.tmp_shestopalov_acq_trx")
assert fact.iloc[0, 0] > 0, "Факт пуст: сначала выполните 01_te_trx.ipynb"

with imp:
    imp.execute("set MEM_LIMIT=8g")
    imp.execute("drop table if exists sandbox_ai.tmp_shestopalov_acq_terminals")
    imp.execute("""
create table sandbox_ai.tmp_shestopalov_acq_terminals
as
with terminals as (
  select
    cast(t.c_nter as string) as c_nter,
    t.d_ter_install,
    t.d_ter_close,
    row_number() over (
      partition by cast(t.c_nter as string)
      order by
        coalesce(cast(t.d_ter_install as date), cast('1900-01-01' as date)) desc,
        coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) desc,
        cast(t.c_pos_serial as string) desc
    ) as rn
  from ods_alpha.scd1_pos_terminals t
  where cast(t.c_nter as string) in (select id_pos_terminal from sandbox_ai.tmp_shestopalov_acq_trx)
    and coalesce(t.ods_deleted_flg, '0') <> '1'
),
device as (
  select
    case when trim(cast(id_device as string)) rlike '^[0-9]+$'
         then cast(cast(trim(cast(id_device as string)) as bigint) as string)
         else upper(trim(cast(id_device as string))) end as id_norm,
    case when trim(cast(code_device as string)) rlike '^[0-9]+$'
         then cast(cast(trim(cast(code_device as string)) as bigint) as string)
         else upper(trim(cast(code_device as string))) end as code_norm,
    trim(cast(model_device as string)) as model_device
  from dwh.bmrt_bm_det_device
  where nullif(trim(cast(model_device as string)), '') is not null
    and lower(trim(cast(model_device as string))) not in ('none', 'nan', 'null', '<null>', 'nat')
),
by_id as (
  select id_norm, max(model_device) as model_device from device where id_norm is not null group by id_norm
),
by_code as (
  select code_norm, max(model_device) as model_device from device where code_norm is not null group by code_norm
)
select
  t.c_nter as id_pos_terminal,
  coalesce(i.model_device, c.model_device) as version,
  t.d_ter_install as date_on,
  t.d_ter_close as date_off
from terminals t
left join by_id i
  on i.id_norm = case when trim(t.c_nter) rlike '^[0-9]+$'
                      then cast(cast(trim(t.c_nter) as bigint) as string)
                      else upper(trim(t.c_nter)) end
left join by_code c
  on c.code_norm = case when trim(t.c_nter) rlike '^[0-9]+$'
                        then cast(cast(trim(t.c_nter) as bigint) as string)
                        else upper(trim(t.c_nter)) end
where t.rn = 1
""")
print("собрано sandbox_ai.tmp_shestopalov_acq_terminals")


In [ ]:
with imp:
    display(imp.fetch("""
select
  count(*) as rows_cnt,
  count(distinct id_pos_terminal) as terminal_cnt,
  sum(case when version is null then 1 else 0 end) as version_null,
  sum(case when date_off is null then 1 else 0 end) as open_cnt
from sandbox_ai.tmp_shestopalov_acq_terminals
"""))
    display(imp.fetch("select * from sandbox_ai.tmp_shestopalov_acq_terminals limit 20"))
